# Refinamento — Selo Ambiental 2026 (SEMARH-PI)

Transforma o dado **bruto** do Selo Ambiental num dataset **pronto para o painel**.

| | |
|---|---|
| **Origem** | `s3a://entrada/sermarh_painel/selo_ambiental_2026.parquet` (a mesma que o Dremio expõe em `coleta."semarh_painel"`) |
| **Destino** | `nessie.refinamento.semarh_painel` (resultado final) · `..._fases` (as 3 fases) · dois resumos |
| **Grão** | principal: 1 linha por município (os 224 do Piauí) · fases: 1 linha por município × fase (672) |

**Fases 1/2/3** são as fases de avaliação/recurso da edição 2026. A **fase 3 é o resultado
final** (consolidado) e alimenta a tabela principal. As três vão para
`semarh_painel_fases`, porque o painel antigo da SEMARH (Looker Studio) publica a **fase 1** —
sem elas é impossível explicar por que os dois painéis mostram números diferentes.

O `resultado` de cada município é um de: **Selo A/B/C**, **Não elegível** ou **Não habilitado**.
A coluna derivada `situacao` agrupa isso em: *Com selo*, *Não elegível*, *Não habilitado* e
*Não postulado* (esta fica em 0 nesta edição — todos os 224 municípios postularam).

Duas correções que o refinamento aplica sobre a fonte:

- `cod_ibge` (vem de `aux4`): código IBGE do município sem o prefixo do estado —
  `2200000 + cod_ibge`. É a chave estável para join; o nome não serve (a dimensão da casa
  grava `Nazária do Piauí` onde a fonte grava `Nazária`).
- **Município não habilitado fica sem apuração** (`NULL`), não com `0`. A fonte grava `0` em
  `CRITÉRIOS 1` para os 5 inabilitados — misturar isso com "atendeu zero critérios" inflava o
  primeiro balde do gráfico de critérios (é o que o painel antigo mostra: 14 em vez de 9 + 5).


## 1. Ler o dado bruto (camada de entrada, no MinIO)


In [1]:
from pyspark.sql import functions as F
from lakehouse import sessao, ler_arquivo, gravar, perfil, listar

spark = sessao("refinamento-selo-ambiental")

# ler_arquivo usa a zona de entrada por padrao -> s3a://entrada/<caminho>
bruto = ler_arquivo(spark, "sermarh_painel/selo_ambiental_2026.parquet")
print("linhas brutas:", bruto.count())
bruto.printSchema()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
software.amazon.awssdk#url-connection-client added as a dependency
org.postgresql#postgresql added as a dependency
com.mysql#mysql-connector-j added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c6406e16-eaf7-4640-91dc-760331a60a37;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.9.2 in central
	found org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12;0.106.0 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in c

linhas brutas: 224
root
 |-- MUNICÍPIO: string (nullable = true)
 |-- PROCESSO: string (nullable = true)
 |-- HABILITADO 1: string (nullable = true)
 |-- HABILITADO 2: string (nullable = true)
 |-- HABILITADO 3: string (nullable = true)
 |-- PONTOS 1: double (nullable = true)
 |-- PONTOS 2: double (nullable = true)
 |-- PONTOS 3: double (nullable = true)
 |-- CRITÉRIOS 1: long (nullable = true)
 |-- CRITÉRIOS 2: double (nullable = true)
 |-- CRITÉRIOS 3: double (nullable = true)
 |-- RESULTADO 1: string (nullable = true)
 |-- RESULTADO 2: string (nullable = true)
 |-- RESULTADO 3: string (nullable = true)
 |-- aux2: string (nullable = true)
 |-- aux3: string (nullable = true)
 |-- pactos: string (nullable = true)
 |-- aux4: long (nullable = true)



## 2. Refinar

Renomeia para nomes limpos (sem acento/espaço), tipa, deriva `selo`, `situacao`,
`pacto_ambiental` e extrai `latitude`/`longitude` do campo de coordenadas.

A derivação vira uma função `refinar(fase)` — as três fases usam exatamente a mesma
regra, mudando só o sufixo das colunas de origem. Daí saem os dois destinos: `ref`
(fase 3, resultado final) e `fases` (as três, formato longo).


In [ ]:
from functools import reduce

from pyspark.sql import DataFrame

# aux3 = "lat, lon"  ->  remove espacos e divide na virgula
coord = F.split(F.regexp_replace(F.col("aux3"), " ", ""), ",")


def refinar(fase: int) -> DataFrame:
    """Refina UMA fase. A derivacao e identica nas tres; muda so o sufixo."""
    res = F.upper(F.trim(F.col(f"`RESULTADO {fase}`")))
    habilitado = F.upper(F.trim(F.col(f"`HABILITADO {fase}`"))) == "SIM"
    # Nao habilitado nao tem apuracao. A fonte grava 0 em `CRITÉRIOS 1` (e vazio
    # nas fases 2 e 3): "atendeu zero criterios" e "nao foi avaliado" sao
    # coisas diferentes. Normaliza para NULL, senao o painel soma os dois no
    # mesmo balde do grafico de criterios.
    pontos = F.col(f"`PONTOS {fase}`").cast("double")
    criterios = F.col(f"`CRITÉRIOS {fase}`").cast("int")
    return (
        bruto.select(
            F.trim(F.col("`MUNICÍPIO`")).alias("municipio"),
            F.trim(F.col("PROCESSO")).alias("processo"),
            # aux4 e o codigo IBGE do municipio sem o prefixo do estado:
            # codigo completo = 2200000 + cod_ibge. Chave estavel para join —
            # o nome nao serve (a dimensao da casa grava "Nazária do Piauí").
            F.col("aux4").cast("int").alias("cod_ibge"),
            F.trim(F.col("aux2")).alias("territorio_desenvolvimento"),
            habilitado.alias("habilitado"),
            F.when(habilitado, pontos).alias("pontos"),
            F.when(habilitado, criterios).alias("criterios_atendidos"),
            res.alias("_res"),
            (F.lower(F.trim(F.col("pactos"))) == "sim").alias("pacto_ambiental"),
            coord.getItem(0).cast("double").alias("latitude"),
            coord.getItem(1).cast("double").alias("longitude"),
        )
        .withColumn("resultado",
            F.when(F.col("_res") == "SELO A", "Selo A")
             .when(F.col("_res") == "SELO B", "Selo B")
             .when(F.col("_res") == "SELO C", "Selo C")
             .when(F.col("_res") == "NÃO ELEGÍVEL", "Não elegível")
             .when(F.col("_res") == "NÃO HABILITADO", "Não habilitado"))
        .withColumn("selo", F.regexp_extract(F.col("_res"), r"SELO ([ABC])", 1))
        .withColumn("selo", F.when(F.col("selo") == "", None).otherwise(F.col("selo")))
        .withColumn("tem_selo", F.col("_res").startswith("SELO"))
        .withColumn("situacao",
            F.when(F.col("_res").startswith("SELO"), "Com selo")
             .when(F.col("_res") == "NÃO ELEGÍVEL", "Não elegível")
             .when(F.col("_res") == "NÃO HABILITADO", "Não habilitado")
             .otherwise("Não postulado"))
        .drop("_res")
        .select(
            "municipio", "processo", "cod_ibge", "territorio_desenvolvimento",
            "situacao", "resultado", "selo", "tem_selo", "pontos",
            "criterios_atendidos", "habilitado", "pacto_ambiental",
            "latitude", "longitude",
        )
    )


# Tabela principal: o resultado FINAL (fase 3). Grao: 1 linha por municipio.
ref = refinar(3)

# Historico das tres fases. Grao: 1 linha por municipio x fase (224 x 3).
# E o que permite reproduzir o painel antigo da SEMARH (publicado na fase 1)
# e comparar a evolucao entre as fases sem reprocessar nada.
fases = reduce(
    DataFrame.unionByName,
    [refinar(n).withColumn("fase", F.lit(n)) for n in (1, 2, 3)],
).select("fase", "municipio", "processo", "cod_ibge", "territorio_desenvolvimento",
         "situacao", "resultado", "selo", "tem_selo", "pontos", "criterios_atendidos",
         "habilitado", "pacto_ambiental", "latitude", "longitude")

perfil(ref)
print("\nfases (formato longo):", fases.count(), "linhas")
fases.groupBy("fase").pivot("resultado").count().orderBy("fase").show(truncate=False)

## 3. Validar antes de gravar


In [ ]:
regras = {
    "224 municípios":        ref.count() == 224,
    "município único":       ref.select("municipio").distinct().count() == ref.count(),
    "código IBGE único":     ref.select("cod_ibge").distinct().count() == ref.count(),
    "toda linha com coord":  ref.filter("latitude IS NULL OR longitude IS NULL").count() == 0,
    "situação preenchida":   ref.filter("situacao IS NULL").count() == 0,
    "selo só quando tem":    ref.filter("tem_selo = true AND selo IS NULL").count() == 0,
    "3 fases x 224":       fases.count() == 672,
    "fase completa":       fases.groupBy("fase").count().filter("count <> 224").count() == 0,
    # Nao habilitado nao pode ter apuracao (era 0 na fase 1, virou NULL).
    "sem apuração se inabilitado":
        fases.filter("habilitado = false AND criterios_atendidos IS NOT NULL").count() == 0,
}
for regra, passou in regras.items():
    print(f"  {'OK  ' if passou else 'FALHOU'}  {regra}")

# A regra da edicao: o selo sai do NUMERO DE CRITERIOS, nao da pontuacao.
# Nao vira assert (a fonte manda), mas o que destoa tem de aparecer aqui.
faixa = (F.when(F.col("criterios_atendidos") >= 6, "Selo A")
          .when(F.col("criterios_atendidos") >= 4, "Selo B")
          .when(F.col("criterios_atendidos") == 3, "Selo C")
          .when(F.col("criterios_atendidos") >= 0, "Não elegível"))
fora = fases.filter(F.col("criterios_atendidos").isNotNull() & (faixa != F.col("resultado")))
print(f"\n  municípios fora da regra critérios->selo: {fora.count()}")
fora.select("fase", "municipio", "criterios_atendidos", "resultado", "pontos").show(truncate=False)

assert all(regras.values()), "corrija antes de gravar"

## 4. Gravar a tabela principal em `refinamento`


In [ ]:
gravar(ref,   "refinamento.semarh_painel",       modo="substituir")
gravar(fases, "refinamento.semarh_painel_fases", modo="substituir")

# A tabela chamava-se `_rodadas` ate a nomenclatura virar "fase" (a mesma do
# painel da SEMARH: 1a/2a/3a fase). Remove a antiga para nao ficarem duas
# versoes do mesmo dado no catalogo.
spark.sql("DROP TABLE IF EXISTS nessie.refinamento.semarh_painel_rodadas")

## 5. Resumos para o painel

Dois recortes que o dashboard consome direto: municípios **por tipo de selo/situação** e
**por número de critérios atendidos**.


In [5]:
por_selo = (
    ref.groupBy("resultado")
       .agg(F.count("*").alias("n_municipios"))
       .orderBy("resultado")
)
por_criterios = (
    ref.groupBy("criterios_atendidos")
       .agg(F.count("*").alias("n_municipios"))
       .orderBy("criterios_atendidos")
)
gravar(por_selo,      "refinamento.semarh_painel_por_selo",      modo="substituir")
gravar(por_criterios, "refinamento.semarh_painel_por_criterios", modo="substituir")

  nessie.refinamento.semarh_painel_por_selo: 5 linhas  ->  s3a://armazem/refinamento/
  nessie.refinamento.semarh_painel_por_criterios: 11 linhas  ->  s3a://armazem/refinamento/


'nessie.refinamento.semarh_painel_por_criterios'

## 6. Conferir o resultado


In [ ]:
listar(spark, "refinamento")
print("\nMunicípios por situação (resultado final):")
ref.groupBy("situacao").count().orderBy(F.desc("count")).show(truncate=False)
print("Municípios por tipo de selo (resultado final):")
por_selo.show(truncate=False)
print("Evolução entre as fases (a fase 1 é a publicada no painel antigo):")
(fases.groupBy("resultado").pivot("fase", [1, 2, 3]).count()
        .orderBy("resultado").show(truncate=False))
print("Pontuação e resultado (amostra, maiores pontuações):")
ref.select("municipio", "resultado", "criterios_atendidos", "pontos") \
   .orderBy(F.desc("pontos")).show(10, truncate=False)